# Scaling Laws

This notebook contains a set of experiments to discover scaling laws on a character-tokenized dataset and a small language model based on the encoder-only transformer architecture.

References:

- https://arxiv.org/abs/2001.08361

- https://arxiv.org/pdf/2203.15556

- https://arxiv.org/html/2406.12907v2

- https://github.com/BenAgro314/MinChilla?tab=readme-ov-file

- https://arxiv.org/abs/2305.07759

- https://huggingface.co/datasets/HuggingFaceTB/cosmopedia

## Utils

### Model definition

In [ ]:
import torch.nn as nn
import torch
import math
from dataclasses import dataclass
torch.set_float32_matmul_precision('high') # use TF32 for matmul operations to speed up GPU operations (note bandwidth is still the bottleneck)


torch.manual_seed(1233)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


@dataclass
class ModelConfig:
    vocab_size: int = 64
    n_embedding: int = 1024
    n_layers: int = 2
    n_heads: int = 4
    context_window: int = 128


class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding from "Attention is All You Need".
    """
    def __init__(self, n_embedding, context_window):
        super().__init__()
        self.pe = torch.zeros(context_window, n_embedding)
        position = torch.arange(0, context_window, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, n_embedding, 2).float() * (-math.log(10000.0) / n_embedding)
        )
        self.pe[:, 0::2] = torch.sin(position * div_term)
        self.pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = self.pe.unsqueeze(0)  # shape: (1, context_window, n_embedding)

    def forward(self, x):
        """
        x: shape [batch_size, seq_len, n_embedding]
        """
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :].to(x.device)


class SmallTransformer(nn.Module):
    """
    Small Transformer Based Model
    """

    def __init__(self, model_config: ModelConfig):

        super().__init__()

        # model hyper-parameters
        self.vocab_size = model_config.vocab_size
        self.n_embedding = model_config.n_embedding
        self.n_layers = model_config.n_layers
        self.n_heads = model_config.n_heads
        self.context_window = model_config.context_window

        # embeddings
        self.token_embedding = nn.Embedding(num_embeddings=self.vocab_size, embedding_dim=self.n_embedding)
        self.positional_embedding = PositionalEncoding(context_window=self.context_window, n_embedding=self.n_embedding)

        # N transformer layers
        dim_feedforward = self.n_embedding * 4
        encoder_layer = nn.TransformerEncoderLayer(nhead=self.n_heads, d_model=self.n_embedding, dim_feedforward=dim_feedforward, batch_first=True)
        layer_norm = nn.LayerNorm(self.n_embedding)
        self.transformer_blocks = nn.TransformerEncoder(encoder_layer=encoder_layer, num_layers=self.n_layers, norm=layer_norm)

        # Linear Head
        self.lm_head = nn.Linear(self.n_embedding, self.vocab_size)

        # weight sharing schema
        self.token_embedding.weight = self.lm_head.weight

    def forward(self, x):
        """
        x: shape [batch_size, seq_len, n_embedding]
        """
        out = self.token_embedding(x)
        out = self.positional_embedding(out)
        out = self.transformer_blocks(out)
        out = self.lm_head(out)
        return out

### Data Processing

In [ ]:
from torch.utils.data import Dataset, DataLoader, Sampler


class CustomDataset(Dataset):
    """
    Creates a token-level dataset by slicing a single long token tensor.
    Returns (x, y) pairs where x and y are arrays of integer IDs for tokens.
    Tokens are read from a memory-mapped file for efficiency.
    """
    def __init__(self, mmap_tokens, seq_len):
        super().__init__()
        self.tokens = mmap_tokens
        self.seq_len = seq_len

    def __len__(self):
        return int((len(self.tokens) - 1) / self.seq_len)

    def __getitem__(self, idx):
        i = idx * self.seq_len
        x = torch.from_numpy(self.tokens[i : i + self.seq_len].astype('long'))
        y = torch.from_numpy(self.tokens[i + 1 : i + self.seq_len + 1].astype('long'))
        return x, y


from time import time
import numpy as np
import os
from itertools import chain


def create_custom_dataset(huggingface_dataset, tokenizer, tokens_dir: str, context_window: int):

    # Define a function to tokenize each example for the map function
    # def tokenize_function(example):
    #     return {'input_ids': tokenizer.encode(example['text'])}

    MAX_LEN = 128

    eos = tokenizer.eos_token_id

    def tokenize_batch(batch):
        out = tokenizer(batch["text"], padding=False, truncation=False, add_special_tokens=False)["input_ids"]
        out = [ids + [eos] for ids in out]
        return {"input_ids": out}

    t0 = time()

    # Apply the tokenization in parallel using datasets.map
    tokenized_data = huggingface_dataset.map(
        tokenize_batch,
        remove_columns=huggingface_dataset.column_names, # Remove original columns to save memory
        batched=True,
        batch_size=5000,
        num_proc=os.cpu_count()
    ).with_format('numpy')
    print(f"Data tokenization took {time()-t0:.2f} seconds")

    # concat tokens all in one array
    t0 = time()
    all_ids = list(chain.from_iterable(tokenized_data["input_ids"]))
    tokens_np = np.asarray(all_ids, dtype=np.int32)

    print(f"Concat tokens took {time()-t0:.2f} seconds")

    # save numpy array as binary file and do memory-mapping
    os.makedirs("data", exist_ok=True)
    tokens_np.tofile(tokens_dir)
    tokens_memmap = np.memmap(
        tokens_dir,
        dtype=np.int32,
        mode="r"
    )

    # create dataset which access to tokens using memory-mapping
    dataset = CustomDataset(tokens_memmap, seq_len=model_config.context_window)

    return dataset


### Training & Eval utils

In [ ]:

from torch.optim import AdamW
import torch.nn.functional as F


@dataclass
class TrainingConfig:
    lr: float = 3e-4
    batch_size: int = 4096 # 128


def criterion(y, targets):
    """
    Compute Cross Entropy
    """
    loss = F.cross_entropy(y, targets, ignore_index=0)
    return loss


class EarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience
        self.best = float("inf")
        self.bad = 0

    def step(self, val_loss, model):
        if val_loss < self.best:
            self.best = val_loss
            self.bad = 0
        else:
            self.bad += 1

        return self.bad >= self.patience


from tqdm import tqdm


def train_num_iterations(optimizer, scheduler, model, train_data_loader: DataLoader, validation_data_loader: DataLoader, max_num_iterations: int, writer):

    progress_bar = tqdm(total=max_num_iterations, desc=f"[Train]")

    num_iterations = 0
    early_stopping = EarlyStopping()

    train_data_loader_iter = iter(train_data_loader)

    while num_iterations < max_num_iterations:

        # activate train mode
        model.train()

        try:
            x, y = next(train_data_loader_iter)
        except StopIteration:
            train_data_loader_iter = iter(train_data_loader)
            x, y = next(train_data_loader_iter)

        x = x.to(device)
        y = y.to(device)

        # remove old gradients
        optimizer.zero_grad()

        # compute loss
        logits = model(x)
        train_loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))

        # compute validation loss & check early stopping
        if num_iterations % 10 == 0:
          val_loss = evaluate(model, validation_data_loader, num_iterations=num_iterations, writer=writer)
          # if early_stopping.step(val_loss, model):
          #   print(f"Early stopping triggered at iteration {num_iterations}")
          #   break

        # compute gradients
        train_loss.backward()

        # clip gradient
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # prevent model from bigger shots on gradient from batches

        # back-propagate
        optimizer.step()

        # Step the scheduler if provided
        if scheduler:
            scheduler.step()
            current_lr = scheduler.get_last_lr()[0]
            writer.add_scalar("Learning Rate/Step", current_lr, num_iterations)


        # log
        writer.add_scalar("Loss/Train_iter", train_loss.item(), num_iterations)

        num_iterations += 1
        progress_bar.update(1)


@torch.no_grad()
def evaluate(model, validation_data_loader, num_iterations: int, writer):

    # activate eval mode
    model.eval()

    # progress_bar = tqdm(enumerate(validation_data_loader), total=len(validation_data_loader), desc=f"Epoch {num_iterations+1} [Validation]")

    epoch_loss = 0.0

    for batch_idx, (x, y) in enumerate(validation_data_loader):

        x = x.to(device)
        y = y.to(device)

        # compute loss
        logits = model(x)
        val_loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        epoch_loss += val_loss

    epoch_loss = epoch_loss / len(validation_data_loader)
    writer.add_scalar("Loss/Val_iter", epoch_loss.item(), num_iterations)

    return epoch_loss


def get_param_count(model: SmallTransformer):
    return sum([p.numel() for p in model.parameters()])

def convert_flops_to_tokens(flops: int, param_count: int) -> int:
    num_tokens = int(flops / (6 * param_count))
    return num_tokens

def convert_num_tokens_to_training_steps(num_tokens, seq_len, batch_size):
    steps = num_tokens / (seq_len * batch_size)
    return int(steps)


## Figures

In [ ]:


import os
from pathlib import Path
from typing import Iterable, Optional, Sequence

import pandas as pd

from tensorboard.backend.event_processing import event_accumulator


def extract_scalars_from_tb(
    logdir: str | os.PathLike,
    tags: Optional[Sequence[str]] = None,
    recursive: bool = True,
) -> pd.DataFrame:
    """
    Extract scalar time series from TensorBoard event files under `logdir`.

    Args:
        logdir: Path that contains TensorBoard event files (e.g., outputs/exp/logs).
        tags: If provided, only extract these scalar tags (exact match).
              If None, extract all scalar tags found.
        recursive: Whether to scan subdirectories for event files.

    Returns:
        DataFrame with columns: ["run", "tag", "step", "wall_time", "value"]
    """
    logdir = Path(logdir)

    # TensorBoard expects a "run directory" that contains one or more event files.
    # We treat each directory that has event files as a separate "run".
    pattern = "**/events.out.tfevents.*" if recursive else "events.out.tfevents.*"
    event_files = list(logdir.glob(pattern))

    if not event_files:
        raise FileNotFoundError(f"No TensorBoard event files found under: {logdir}")

    run_dirs = sorted({f.parent for f in event_files})

    rows: list[dict] = []

    for run_dir in run_dirs:
        # Load events for this run directory
        ea = event_accumulator.EventAccumulator(
            str(run_dir),
            size_guidance={event_accumulator.SCALARS: 0},  # 0 = load all scalars
        )
        ea.Reload()

        available_tags = ea.Tags().get("scalars", [])
        if not available_tags:
            continue

        wanted_tags: Iterable[str]
        if tags is None:
            wanted_tags = available_tags
        else:
            # Keep only tags that exist in this run
            wanted_tags = [t for t in tags if t in available_tags]

        for tag in wanted_tags:
            for ev in ea.Scalars(tag):
                rows.append(
                    {
                        "run": str(run_dir.relative_to(logdir)),
                        "tag": tag,
                        "step": ev.step,
                        "wall_time": ev.wall_time,
                        "value": ev.value,
                    }
                )

    if not rows:
        raise RuntimeError(
            f"No scalar data found. Check that scalars were logged and logdir is correct: {logdir}"
        )

    df = pd.DataFrame(rows).sort_values(["run", "tag", "step"]).reset_index(drop=True)
    return df


import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

def plot_all_loss_curves(logdir: str):
    """
    Plots the training and validation loss curves for all runs from TensorBoard logs.

    Args:
        logdir: The base directory containing TensorBoard log subdirectories.
    """
    df = extract_scalars_from_tb(Path(logdir), tags=["Loss/Train_iter", "Loss/Val_iter"], recursive=True)

    if df.empty:
        print("No loss data found to plot.")
        return

    # Group by run to plot each experiment separately
    for run_name, run_df in df.sort_values('run').groupby('run'):
        plt.figure(figsize=(12, 6))

        train_loss_df = run_df[run_df['tag'] == 'Loss/Train_iter'].sort_values('step')
        val_loss_df = run_df[run_df['tag'] == 'Loss/Val_iter'].sort_values('step')

        if not train_loss_df.empty:
            plt.plot(train_loss_df['step'], train_loss_df['value'], label='Training Loss', alpha=0.8)
        if not val_loss_df.empty:
            plt.plot(val_loss_df['step'], val_loss_df['value'], label='Validation Loss', alpha=0.8)

        plt.title(f'Loss Curves for Run: {run_name}')
        plt.xlabel('Training Step')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)
        # plt.yscale('log') # Loss often benefits from a log scale
        plt.tight_layout()
        plt.show()
        plt.close() # Close the figure to free memory



## Main Code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Download, Tokenize

In [ ]:
change_data_flag = False # True
experiment = "scaling_laws_cosmopedia_16k/" #"/scaling_laws"

In [ ]:
from datasets import load_dataset, Dataset
import os

# -----------------------
if change_data_flag:
  dataset = "HuggingFaceTB/cosmopedia"
  train_data = load_dataset(dataset, "auto_math_text", split="train", num_proc=os.cpu_count()) # [:60%]
  validation_data_iterable = load_dataset(dataset, "web_samples_v2", split="train", streaming=True).take(1000) # 10%
  validation_data = Dataset.from_generator(lambda: validation_data_iterable)
  print("data downloaded!")

data downloaded!
Built vocabulary with 90 characters.


In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.normalizers import NFKC
from transformers import PreTrainedTokenizerFast
import os


# --- config ---
TEXT_COL = "text"
VOCAB_SIZE = 16000
OUT_DIR = "drive/MyDrive/scaling_laws_cosmopedia_16k/tokenizer/bpe_16k"


if change_data_flag:

  os.makedirs(OUT_DIR, exist_ok=True)

  SPECIAL_TOKENS = ["<pad>", "<unk>", "<bos>", "<eos>"]

  # --- load text ---
  tokenizer_dataset = train_data.select(range(int(1e5)))

  def text_iter():
      for ex in tokenizer_dataset:
          yield ex[TEXT_COL]

  # --- train fast (Rust) tokenizer ---
  tokenizer = Tokenizer(BPE(unk_token="<unk>"))
  tokenizer.normalizer = NFKC()
  tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=True)
  tokenizer.decoder = ByteLevelDecoder()

  trainer = BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS)
  tokenizer.train_from_iterator(text_iter(), trainer=trainer)

  # --- save (fast tokenizer.json + transformers-compatible folder) ---
  tokenizer.save(f"{OUT_DIR}/tokenizer.json")


tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=f"{OUT_DIR}/tokenizer.json",
    unk_token="<unk>", pad_token="<pad>", bos_token="<bos>", eos_token="<eos>",
)
tokenizer.save_pretrained(OUT_DIR)

print("Saved to:", OUT_DIR)
print("Vocab size:", tokenizer.vocab_size)
print(tokenizer("Once upon a time.")["input_ids"])

### Model

In [ ]:
# -----------------------
model_config = ModelConfig(vocab_size=tokenizer.vocab_size, n_embedding=128, n_layers=2, n_heads=2, context_window=128)
print(model_config)
model = SmallTransformer(model_config).to(device)
print(f"Number of model parameters: {sum([l.numel() for l in model.parameters()])/1e6:.2f}")

ModelConfig(vocab_size=90, n_embedding=256, n_layers=2, n_heads=4, context_window=128)
Number of model parameters: 1.10


### Dataset creation

In [ ]:
# ----------------------
import subprocess

local_dir = "data"
drive_dir = "drive/MyDrive/scaling_laws_cosmopedia_16k/data"

local_train_tokens_dir = local_dir + "/train_tokens_np.bin"

local_validation_tokens_dir = local_dir + "/validation_tokens_np.bin"

if change_data_flag:

  # Tokenize
  train_dataset = create_custom_dataset(train_data, tokenizer, tokens_dir=local_train_tokens_dir, context_window=model_config.context_window)
  validation_dataset = create_custom_dataset(validation_data, tokenizer, tokens_dir=local_validation_tokens_dir, context_window=model_config.context_window)
  del train_data, validation_data

  # copy to drive
  result = subprocess.run(["cp", "-r", local_dir, drive_dir])

else:

  # copy from drive
  subprocess.run(["cp", "-r", drive_dir, local_dir])

  # Read from drive
  mmap_tokens = np.memmap(
    filename=local_train_tokens_dir,
    dtype=np.int32,
    mode="r"
  )
  train_dataset = CustomDataset(mmap_tokens, seq_len=model_config.context_window)

  mmap_tokens = np.memmap(
    filename=local_validation_tokens_dir,
    dtype=np.int32,
    mode="r"
  )
  validation_dataset = CustomDataset(mmap_tokens, seq_len=model_config.context_window)

Data tokenization took 0.03 seconds
Concat tokens took 0.52 seconds
Data tokenization took 0.02 seconds
Concat tokens took 0.01 seconds


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

unique, counts = np.unique(train_dataset.tokens, return_counts=True)
print("vocab used:", len(unique), "most common fraction:", counts.max()/counts.sum())
print("top ids:", unique[np.argsort(-counts)][:10])

# Get the indices of the top 10 most frequent tokens
top_10_indices = np.argsort(-counts)[:20]

# Get the actual IDs and their counts
top_10_ids = unique[top_10_indices]
top_10_counts = counts[top_10_indices]

# Calculate the fraction of total tokens for each of the top 10
total_tokens = counts.sum()
top_10_fractions = (top_10_counts / total_tokens) * 100

# Get the actual tokens (strings) for visualization
top_10_tokens = [tokenizer.decode([_id]) for _id in top_10_ids]

# Create the bar plot
plt.figure(figsize=(12, 6))
plt.bar(top_10_tokens, top_10_fractions, color='skyblue')
plt.xlabel('Token')
plt.ylabel('Fraction of Total Tokens (%)')
plt.title('Top 10 Most Frequent Tokens by Fraction of Total')
plt.xticks(rotation=45, ha='right') # Rotate labels for better readability
plt.tight_layout()
plt.show()

In [ ]:
print(f"Tokens in train dataset: {len(train_dataset.tokens)/1e6} M")

### TensorBoard

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir experiments

### Training Loop

In [ ]:
# -----------------------
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm

# -----------------------
model_config = ModelConfig(vocab_size=tokenizer.vocab_size, n_embedding=384, n_layers=4, n_heads=4)

model = SmallTransformer(model_config).to(device)
model = torch.compile(model)
tqdm.write(f"Model config: {model_config}")
tqdm.write(f"Number of model parameters: {sum([l.numel() for l in model.parameters()])/1e6:.2f}")

training_config = TrainingConfig(batch_size=int(2**9))

flops = int(1e17)
num_tokens = convert_flops_to_tokens(flops=flops, param_count=get_param_count(model))
max_num_iterations = convert_num_tokens_to_training_steps(
    num_tokens=num_tokens,
    seq_len=model_config.context_window,
    batch_size=training_config.batch_size
)

optimizer = AdamW(model.parameters(), lr=training_config.lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_num_iterations, eta_min=0.1*training_config.lr)

# -----------------------
train_data_loader = DataLoader(
    train_dataset,
    batch_size=training_config.batch_size
)
validation_data_loader = DataLoader(
    validation_dataset,
    batch_size=training_config.batch_size
)

# -----------------------
log_file_name = "experiments/dummy_experiment_2/logs"
writer = SummaryWriter(log_file_name)

writer.add_scalar("Model/Total Parameters", get_param_count(model), 0)

train_num_iterations(
    optimizer=optimizer,
    scheduler=scheduler,
    model=model,
    train_data_loader=train_data_loader,
    validation_data_loader=validation_data_loader,
    max_num_iterations=max_num_iterations,
    writer=writer,
)


Epoch 1 [Validation]: 100%|██████████| 9/9 [00:02<00:00,  4.11it/s]


In [8]:
# !tensorboard --logdir ./experiments

### Experiments Design

In [ ]:
model_sizes = range(64, 64*7, 64)# range(64, 32*5, 32)  #range(128, 64*15, 64)

flops = 1e15
batch_size = int(2**9)

for n_embedding in model_sizes:

      # -----------------------
      # define model config
      n_heads = 4 # max(n_embedding // 64, 2)
      n_heads = n_heads if n_heads // 2 != 1 or n_heads == 2 else n_heads - 1
      n_layers = 4 # max(n_embedding // 64, 2)
      n_layers = n_layers if n_layers // 2 != 1 or n_layers == 2 else n_layers - 1

      model_config = ModelConfig(
          vocab_size=16000,
          n_embedding=n_embedding,
          n_layers=n_layers,
          n_heads=n_heads,
          context_window=128
      )

      model = SmallTransformer(model_config).to(device)
      params_count = get_param_count(model)
      num_tokens = convert_flops_to_tokens(flops=flops, param_count=params_count)
      max_num_iterations = convert_num_tokens_to_training_steps(
          num_tokens=num_tokens,
          seq_len=model_config.context_window,
          batch_size=batch_size
      )

      print(f"Model config: {model_config} | Total Parameters: {params_count/1e6} M | Tokens: {num_tokens/1e6} M | Tokens/Parameter: {num_tokens/params_count} | max_num_iterations: {max_num_iterations} | batch_size: {batch_size}")

/home/rubenbalbastre/Escritorio/Repositorios/LLM/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(


Experiment: experiments/experiment_1_32/logs


Epoch 1 [Train]:   0%|          | 0/510 [00:00<?, ?it/s]

Epoch 1 [Validation]: 100%|██████████| 9/9 [00:00<00:00, 49.95it/s]


Experiment: experiments/experiment_1_96/logs


Epoch 1 [Train]:  28%|██▊       | 142/510 [01:49<05:23,  1.14it/s]

### Experiments Run

In [ ]:
# -----------------------
# N Experiments

from torch.utils.tensorboard import SummaryWriter

flop_counts = [
    3e14,
    1e15,
    2e15,
    # 1e16,
    # 1e17
]
model_sizes = range(64, 64*7, 64) # range(128, 64*10, 64)# range(64, 32*5, 32)  #range(128, 64*15, 64)
batch_size = int(2**9)

from tqdm.notebook import tqdm

print(f"Model sizes: {list(model_sizes)}")

for flops in flop_counts:

    for n_embedding in model_sizes:

        log_file_name = f"experiments/experiment_{int(flops/1e13)}_{n_embedding}/logs"
        tqdm.write(f"Experiment: {log_file_name}")
        if not os.path.isdir(log_file_name):

            # -----------------------
            # define model config
            # n_heads = max(n_embedding // 64, 2)
            # n_heads = n_heads if n_heads // 2 != 1 or n_heads == 2 else n_heads - 1
            # n_layers = max(n_embedding // 64, 2)
            # n_layers = n_layers if n_layers // 2 != 1 or n_layers == 2 else n_layers - 1
            n_heads = 4
            n_layers = 4

            model_config = ModelConfig(
                vocab_size=tokenizer.vocab_size,
                n_embedding=n_embedding,
                n_layers=n_layers,
                n_heads=n_heads,
                context_window=128
            )

            model = SmallTransformer(model_config).to(device)
            model = torch.compile(model)
            params_count = get_param_count(model)

            # -----------------------
            # Training config
            training_config = TrainingConfig(batch_size=batch_size)

            # -----------------------
            # Training data
            num_tokens = convert_flops_to_tokens(flops=flops, param_count=params_count)
            max_num_iterations = convert_num_tokens_to_training_steps(
                num_tokens=num_tokens,
                seq_len=model_config.context_window,
                batch_size=training_config.batch_size
            )
            tokens_per_parameter = num_tokens/params_count

            tqdm.write(f"Model config: {model_config} | Total Parameters: {params_count/1e6} M | Tokens: {num_tokens/1e6} M | Tokens/Parameter: {tokens_per_parameter} | flops: {int(flops/1e15)}e15 | max_num_iterations: {max_num_iterations} | batch_size: {batch_size}")

            if max_num_iterations < 100 or tokens_per_parameter > 1e3:
              tqdm.write(f"Skipping: {model_config} | max_num_iterations: {max_num_iterations} | tokens_per_parameter: {tokens_per_parameter}")
              continue

            optimizer = AdamW(model.parameters(), lr=training_config.lr)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_num_iterations, eta_min=0.1*training_config.lr)

            train_data_loader = DataLoader(
                train_dataset,
                batch_size=training_config.batch_size
            )
            validation_data_loader = DataLoader(
                validation_dataset,
                batch_size=training_config.batch_size
            )

            # -----------------------
            writer = SummaryWriter(log_file_name)

            writer.add_scalar("Model/Total Parameters", params_count, 0)
            writer.add_scalar("Model/N Embedding", n_embedding, 0)
            writer.add_scalar("Model/N Heads", n_heads, 0)
            writer.add_scalar("Model/N Layers", n_layers, 0)
            writer.add_scalar("Training/Tokens", num_tokens, 0)
            writer.add_scalar("Training/Batch Size", training_config.batch_size, 0)

            train_num_iterations(
                optimizer=optimizer,
                scheduler=scheduler,
                model=model,
                train_data_loader=train_data_loader,
                validation_data_loader=validation_data_loader,
                max_num_iterations=max_num_iterations,
                writer=writer,
            )


### Figures

In [ ]:
from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import RANSACRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from collections import defaultdict


# Example: point to the directory you pass to SummaryWriter(log_dir=...)
LOGDIR = "experiments"  # or "outputs/experiment-.../logs"

# Put your loss tags here. If you leave tags=None, it extracts all scalar tags.
LOSS_TAGS = None # [
#     "Loss/Train_iter",
#     "Loss/Train_iter_smoothed",
#     "Loss/Val_iter",
#     "Loss/Val_Epoch",
# ]

out_dir = Path(LOGDIR)

df = extract_scalars_from_tb(LOGDIR, tags=None, recursive=True)

required_tags = {"Loss/Val_iter", "Model/Total Parameters"}
run_groups = {run: g for run, g in df.groupby("run")}

flops_to_curve: dict[float, list[tuple[float, float, str]]] = defaultdict(list)

for run_name, run_df in run_groups.items():
    tags = set(run_df["tag"].unique())

    if not required_tags.issubset(tags):
        print(f"Skipping {run_name}: missing tags {required_tags - tags}")
        continue

    train_curve = run_df[run_df["tag"] == "Loss/Val_iter"].sort_values("step")
    model_curve = run_df[run_df["tag"] == "Model/Total Parameters"].sort_values("step")

    final_loss = train_curve.iloc[-1]["value"]
    params = model_curve.iloc[-1]["value"]

    experiment = Path(run_name).parts[0]  # e.g. experiment_1_256
    try:
        _, flops_str, _ = experiment.split("_")
        num_peta_flops = int(flops_str)
        if num_peta_flops >= 3000:
          print(f"Skipping {run_name}: unexpected folder name {experiment}")
          continue
    except ValueError:
        print(f"Skipping {run_name}: unexpected folder name {experiment}")
        continue

    print(f"Adding point: FLOPS={num_peta_flops} Params={params} Loss={final_loss} Experiment={experiment}")
    flops_to_curve[num_peta_flops].append((params, final_loss, experiment))


import os
from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, RANSACRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from collections import defaultdict

import pandas as pd

os.makedirs("resources", exist_ok=True)

min_params = min([v[0] for g in flops_to_curve.values() for v in g])
max_params = max([v[0] for g in flops_to_curve.values() for v in g])
min_flops = min(flops_to_curve.keys())
max_flops = max(flops_to_curve.keys())

print(min_flops, max_flops)
cmap = plt.get_cmap("rainbow")

for k,v in flops_to_curve.items():
    flops_to_curve[k] = sorted(flops_to_curve[k], key=lambda x: x[0])
flops_to_curve_items = sorted(flops_to_curve.items(), key=lambda x: x[0])

minima_params = []
minima_flops = []

# --- Option to choose regressor ---
use_ransac_regressor = False # Set to True to use RANSACRegressor, False for LinearRegression
# ----------------------------------

plt.figure(figsize=(12, 8)) # Added to make the figure bigger

for num_flops, data in flops_to_curve_items: #[(k,v) for k, v in flops_to_curve_items if k == 6]:
    xs_log = np.array([math.log10(x[0]) for x in data])
    ys = np.array([x[1] for x in data])

    X = xs_log.reshape(-1, 1)
    y = ys

    # polynomial fitting
    polynomial_degree = 2

    if use_ransac_regressor:
        model = make_pipeline(
            PolynomialFeatures(degree=polynomial_degree, include_bias=True),
            RANSACRegressor()
        )
    else:
        model = make_pipeline(
            PolynomialFeatures(degree=polynomial_degree, include_bias=True),
            LinearRegression()
        )

    log_normalized_flops = (math.log(num_flops) - math.log(min_flops)) / (
        math.log(max_flops) - math.log(min_flops) + 1e-5
    )
    color = cmap(log_normalized_flops)

    try:
        # Fit the model
        model.fit(X, y)

        if use_ransac_regressor:
            ransac_regressor = model.named_steps['ransacregressor']
            inlier_mask = ransac_regressor.inlier_mask_
            X_fit_points = X[inlier_mask]
            y_fit_points = y[inlier_mask]
            coeffs = ransac_regressor.estimator_.coef_
            intercept = ransac_regressor.estimator_.intercept_
        else:
            linear_regressor = model.named_steps['linearregression']
            X_fit_points = X
            y_fit_points = y
            coeffs = linear_regressor.coef_
            intercept = linear_regressor.intercept_

        c_poly = intercept
        b_poly = coeffs[1]
        a_poly = coeffs[2]

        poly = np.poly1d([a_poly, b_poly, c_poly])

        xs_fit_log = np.linspace(min(xs_log), max(xs_log), 100)
        ys_fit = poly(xs_fit_log)

        xs_fit = 10**xs_fit_log

        plt.scatter(
            10**X_fit_points.flatten(),
            y_fit_points,
            label=f"FLOPs={num_flops}",
            color=color,
            marker='o',
            alpha=0.7
        )

        plt.plot(xs_fit, ys_fit, color=color, linewidth=2)

        if a_poly != 0:
            minima_log = -b_poly / (2 * a_poly)
            minima = 10**minima_log
            y_val = poly(minima_log)
            plt.scatter([minima], [y_val], color=color, marker='x', s=100)

            minima_params.append(minima)
            minima_flops.append(num_flops)
        else:
            print(f"pflops={num_flops}: Quadratic coefficient is zero, cannot compute minima.")

    except ValueError as e:
        regressor_name = "RANSACRegressor" if use_ransac_regressor else "LinearRegression"
        print(f"Skipping {regressor_name} fit for FLOPs={num_flops} due to error: {e}")
        plt.scatter(
            10**xs_log.flatten(),
            ys,
            label=f"FLOPs={num_flops} (no fit)",
            color=color,
            marker='o',
            alpha=0.7
        )


plt.legend(loc='upper center', bbox_to_anchor=(0.5, 1.10),
          ncol=3, fancybox=True, shadow=True)
# plt.yscale("log")
plt.xscale("log")
plt.xlabel("Number of Parameters")
plt.ylabel("Training Loss")
plt.savefig("resources/isoflop_curve.png")
plt.close("all")
plt.show()

# plt.scatter(minima_flops, minima_params)
# m, b = np.polyfit([math.log10(m) for m in minima_flops], [math.log10(m) for m in minima_params], 1)
# plt.plot(minima_flops, [10**(m*math.log10(x) + b) for x in minima_flops], label=f"y={m:.2f}x + {b:.2f}", linestyle="--")
# plt.legend()
# plt.ylabel("Number of Parameters")
# plt.xlabel("Number of PetaFLOPs")
# plt.xscale("log")
# plt.yscale("log")
# plt.savefig("resources/minima_params_vs_flops.png")
# plt.close("all")

# minima_tokens = [(f*1e15)/(6 * p) for f, p in zip(minima_flops, minima_params)]
# plt.scatter(minima_flops, minima_tokens)
# m, b = np.polyfit([math.log10(m) for m in minima_flops], [math.log10(m) for m in minima_tokens], 1)
# plt.plot(minima_flops, [10**(m*math.log10(x) + b) for x in minima_flops], label=f"y={m:.2f}x + {b:.2f}", linestyle="--")
# plt.legend()
# plt.ylabel("Number of Tokens")
# plt.xlabel("Number of PetaFLOPs")
# plt.xscale("log")
# plt.yscale("log")
# plt.savefig("resources/minima_tokens_vs_flops.png")
# plt.close("all")

# plt.scatter(minima_params, minima_tokens)
# m, b = np.polyfit([math.log10(m) for m in minima_params], [math.log10(m) for m in minima_tokens], 1)
# plt.plot(minima_params, [10**(m*math.log10(x) + b) for x in minima_params], label=f"y={m:.2f}x + {b:.2f}", linestyle="--")
# plt.legend()
# plt.ylabel("Number of Tokens")
# plt.xlabel("Number of Params")
# plt.xscale("log")
# plt.yscale("log")
# plt.savefig("resources/minima_tokens_vs_params.png")
# plt.close("all")


import matplotlib.pyplot as plt
import numpy as np
import math


def plot_isoflop_points_only(flops_to_curve_items, min_flops, max_flops):
    #cmap = plt.get_cmap("rainbow")
    plt.figure(figsize=(10, 6))

    for num_flops, data in flops_to_curve_items:
        xs_log = np.array([math.log10(x[0]) for x in data])
        ys = np.array([x[1] for x in data])

        plt.scatter(
            10**xs_log.flatten(),
            ys,
            label=f"FLOPs={num_flops}",
            marker='o',
            alpha=0.7
        )

    plt.legend(loc='upper left', bbox_to_anchor=(0.5, 1.10),
               ncol=3, fancybox=True, shadow=True)
    plt.xscale("log")
    plt.xlabel("Number of Parameters")
    plt.ylabel("Validation Loss")
    plt.title("IsoFLOP Curve (Points Only)")
    plt.savefig("resources/isoflop_curve_points_only.png")
    plt.close()


def plot_minima_without_fitting(flops_to_curve_items):
    minima_params = []
    minima_flops_scaled = [] # These are the scaled flops (e.g., 10, 30, 60)

    for num_flops_scaled, data in flops_to_curve_items:
        min_loss = float('inf')
        best_params = None

        for params, loss, _ in data:
            if loss < min_loss:
                min_loss = loss
                best_params = params

        if best_params is not None:
            minima_params.append(best_params)
            minima_flops_scaled.append(num_flops_scaled)
        else:
            print(f"No valid data found for scaled flops {num_flops_scaled}")

    # Convert scaled flops to actual flops (e.g., 10 -> 1e14) for token calculation
    actual_minima_flops = [f * 1e13 for f in minima_flops_scaled]

    # Plot 1: Minima Parameters vs. Scaled Flops
    plt.figure(figsize=(10, 6))
    plt.scatter(minima_flops_scaled, minima_params)
    if len(minima_flops_scaled) > 1:
        # Fit a linear regression in log-log space for visualization of trend
        m, b = np.polyfit([math.log10(x) for x in minima_flops_scaled], [math.log10(y) for y in minima_params], 1)
        # Plot the fitted line
        plt.plot(minima_flops_scaled, [10**(m*math.log10(x) + b) for x in minima_flops_scaled],
                 label=f"y = 10^({m:.2f}log10(x) + {b:.2f})", linestyle="--", color='red')
        plt.legend()
    plt.ylabel("Number of Parameters")
    plt.xlabel("FLOPs (scaled by 1e13)")
    plt.xscale("log")
    plt.yscale("log")
    plt.title("Minima Parameters vs. FLOPs (Observed Minima)")
    plt.savefig("resources/minima_params_vs_flops_observed.png")
    plt.close()

    # Plot 2: Minima Tokens vs. Scaled Flops
    # Ensure convert_flops_to_tokens function is available (from cell 46040347)
    minima_tokens = [convert_flops_to_tokens(f, p) for f, p in zip(actual_minima_flops, minima_params)]
    plt.figure(figsize=(10, 6))
    plt.scatter(minima_flops_scaled, minima_tokens)
    if len(minima_flops_scaled) > 1:
        m, b = np.polyfit([math.log10(x) for x in minima_flops_scaled], [math.log10(y) for y in minima_tokens], 1)
        plt.plot(minima_flops_scaled, [10**(m*math.log10(x) + b) for x in minima_flops_scaled],
                 label=f"y = 10^({m:.2f}log10(x) + {b:.2f})", linestyle="--", color='red')
        plt.legend()
    plt.ylabel("Number of Tokens")
    plt.xlabel("FLOPs (scaled by 1e13)")
    plt.xscale("log")
    plt.yscale("log")
    plt.title("Minima Tokens vs. FLOPs (Observed Minima)")
    plt.savefig("resources/minima_tokens_vs_flops_observed.png")
    plt.close()

    # Plot 3: Minima Tokens vs. Minima Parameters
    plt.figure(figsize=(10, 6))
    plt.scatter(minima_params, minima_tokens)
    if len(minima_params) > 1:
        m, b = np.polyfit([math.log10(x) for x in minima_params], [math.log10(y) for y in minima_tokens], 1)
        plt.plot(minima_params, [10**(m*math.log10(x) + b) for x in minima_params],
                 label=f"y = 10^({m:.2f}log10(x) + {b:.2f})", linestyle="--", color='red')
        plt.legend()
    plt.ylabel("Number of Tokens")
    plt.xlabel("Number of Parameters")
    plt.xscale("log")
    plt.yscale("log")
    plt.title("Minima Tokens vs. Parameters (Observed Minima)")
    plt.savefig("resources/minima_tokens_vs_params_observed.png")
    plt.close()

# Call the function to plot the observed minima
plot_minima_without_fitting(flops_to_curve_items)
plot_isoflop_points_only(flops_to_curve_items=flops_to_curve_items, min_flops=1e14, max_flops=30e15)